# Phase 4C: one-time paired dev inference
Notebook: https://www.kaggle.com/code/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry
Save a new version of this existing Kaggle notebook. Attach the exact successful version 4 output (scriptVersionId=373428580), the original raw CSV input and the phase-3 SRO dataset. Run once with GPU; never select checkpoints on dev.


In [1]:
"""Kaggle phase-4C dev inference after both validation-selected runs finish."""

import csv
import hashlib
import json
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


BASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")
SRO_DATA = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")
INPUT_NOTEBOOKS = Path("/kaggle/input/notebooks/hungluongduc")
OUT = Path("/kaggle/working/phase4c_paired_dev")
REVISION = "64a8c8eab3e352a784c658aef62be1662607476f"
SPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"
RAW_HASH = "F7EC0B2308EBDC811BFC11828A076F498FD2CB542C503274DAD5CBA852EBD1BC"
SRO_HASH = "2E469335EB7F58C20C9CA1225CB3E4B08338D92FC5B8B750D574C23FED85E302"
FIELDS = ["row_index", "label", "raw_text_sha256", "sro_text_sha256",
          "raw_prob", "sro_prob", "raw_pred", "sro_pred"]
AMBIGUOUS_INDICES = [333, 335, 341, 343, 345, 350, 358, 375, 398, 426, 455, 459, 460, 461, 463, 543, 547, 551, 563, 650, 701, 703, 705, 707, 709, 711, 713, 715, 748, 854, 870, 874, 1035, 1037, 1066, 1078, 1136, 1138, 1140, 1188, 1254, 1300, 1304, 1310, 1333, 1337, 1339, 1343, 1378, 1382, 1399, 1400, 1439, 1447, 1449, 1453, 1455, 1459, 1475, 1477, 1479, 1481, 1483, 1484, 1487, 1488, 1531, 1534, 1537, 1539, 1543, 1545, 1547, 1549, 1574, 1580, 1600, 1616, 1656, 1673, 1675, 1677, 1679, 1681, 1683, 1687, 1723, 1729, 1743, 1747, 1749, 1757, 1838, 1840, 1848, 1857, 1861, 1869, 1873, 1875, 1877, 1879, 1883, 1975, 1976, 2000, 2002, 2016, 2018, 2025, 2033, 2035, 2037, 2039, 2043, 2045, 2047, 2056, 2058, 2062, 2066, 2121, 2125, 2163, 2165, 2171, 2173, 2175, 2179, 2183, 2185, 2197, 2199, 2213, 2215, 2219, 2412, 2418, 2433, 2531, 2534, 2537, 2540, 2543, 2546, 2549, 2552, 2555, 2600, 2603, 2606, 2609, 2612, 2615, 2618, 2621, 2624, 2627, 2630, 2633, 2636, 2639, 2642, 2645, 2648, 2651, 2654, 2657, 2660, 2663, 2730, 2736, 2819, 2822, 2828, 2840, 2843, 2862, 2902, 2903, 2907, 2909, 2917, 2918, 2923, 2929, 2930, 2935, 2936, 2941, 2944, 2945, 2949, 2951, 2953, 2958, 2960, 2962, 2964, 2966, 2975, 2980, 2981, 2989, 2990, 2995, 2996, 3008, 3013, 3014, 3018, 3020, 3024, 3026, 3028, 3029, 3035, 3036, 3038, 3040, 3041, 3044, 3045, 3047, 3049, 3050, 3052, 3053, 3055, 3056, 3057, 3059, 3062, 3065, 3071, 3074, 3096, 3126, 3135, 3202, 3211, 3214, 3221, 3230, 3239, 3242, 3245, 3269, 3275, 3278, 3305, 3314, 3329, 3410, 3413, 3419, 3422, 3425, 3428, 3431, 3434, 3437, 3443, 3446, 3449, 3452, 3455, 3458, 3461, 3464, 3470, 3472, 3473, 3477, 3479, 3500, 3617, 3618, 3627, 3633, 3642, 3645, 3651, 3663, 3675, 3681, 3690, 3696, 3698, 3705, 3711, 3723, 3729, 3733, 3747, 3753, 3762, 3771, 3774, 3789, 3890, 3946, 3950, 3959, 3962, 3965, 3968, 3971, 3974, 3983, 4038, 4059, 4224, 4311, 4315, 4320, 4324, 4328, 4380, 4384, 4403, 4447, 4460, 4461, 4464, 4468, 4470, 4472, 4474, 4476, 4478, 4480, 4484, 4486, 4488, 4492, 4496, 4500, 4504, 4506, 4508, 4512, 4516, 4518, 4520, 4522, 4524, 4528, 4530, 4532, 4534, 4536, 4538, 4540, 4544, 4548, 4564, 4568, 4612, 4616, 4620, 4624, 4628, 4636, 4640, 4644, 4705, 4730, 4734, 4746, 4750, 4754, 4762, 4800, 4804, 4808, 4812, 4816, 4820, 4825, 4828, 4832, 4836, 4840, 4844, 4848, 4851, 4853, 4856, 4860, 4864, 4868, 4871, 4872, 4876, 4882, 4883, 4885, 4889, 4895, 4899, 4903, 4907, 4911, 4915, 4919, 4923, 4927, 4930, 4931, 4935, 4939, 4943, 4947, 4951, 4955, 4959, 4963, 4967, 4971, 4975, 5079, 5083, 5088, 5095, 5096, 5103, 5115, 5119, 5139, 5151, 5155, 5156, 5160, 5167, 5171, 5254, 5258, 5262, 5266, 5270, 5274, 5278, 5282, 5286, 5290, 5294, 5298, 5302, 5306, 5310, 5314, 5318, 5322, 5326, 5330, 5334, 5338, 5342, 5486, 5490, 5494, 5498, 5502, 5506, 5510, 5514, 5518, 5522, 5526, 5557, 5593, 5596, 5602, 5606, 5610, 5618, 5620, 5626, 5628, 5638, 5687, 5777, 5785, 5788, 5801, 5821, 5825, 5829, 5837, 5841, 5849, 5878, 5885, 5893, 5897, 5901, 5912, 5930, 5936, 5943, 5955, 5963, 5973, 5975, 5999, 6000, 6016, 6025, 6026, 6041, 6049, 6058, 6077, 6092, 6116, 6118, 6143, 6151, 6166, 6173, 6185, 6188, 6193, 6215, 6217, 6256, 6283, 6299, 6335, 6338, 6346, 6347, 6352, 6362, 6373, 6380]


def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()


def csv_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if reader.fieldnames != ["text", "label"]:
            raise ValueError(f"invalid CSV header: {path}")
        rows = list(reader)
    if len(rows) != 6404:
        raise ValueError(f"wrong dev row count: {path}")
    return rows


def passage(text):
    prefix, sep = "Sub-question: ", "\n\nPassages:\n"
    if not text.startswith(prefix) or sep not in text:
        raise ValueError("invalid text contract")
    return text.split(sep, 1)[1]


def selected_checkpoint(branch):
    found = list(INPUT_NOTEBOOKS.glob(f"*/phase4c_{branch}/progress.json"))
    if len(found) != 1:
        raise ValueError(f"expected exactly one final {branch} run input, found {len(found)}: {found}")
    progress = json.loads(found[0].read_text(encoding="utf-8"))
    if progress["branch"] != branch or progress["split_sha256"] != SPLIT_HASH or progress["model_revision"] != REVISION:
        raise ValueError(f"{branch} protocol mismatch")
    if progress["raw_sha256"] != "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D":
        raise ValueError("raw train hash mismatch")
    if progress["sro_sha256"] != "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8":
        raise ValueError("SRO train hash mismatch")
    if progress["completed_epoch"] < 10 and progress["early_stopping_patience_counter"] < 3:
        raise ValueError(f"{branch} training has not met the 10-epoch/patience-3 rule")
    expected = {"raw": ("phase4c_raw_epoch4_20261124_02", 4, 10556),
                "sro": ("phase4c_sro_epoch5_20261124_01", 5, 13195)}[branch]
    if (progress["run_id"], progress["completed_epoch"], progress["global_step"]) != expected:
        raise ValueError(f"wrong final {branch} run: {progress['run_id']}")
    name = Path(progress["best_checkpoint"]).name
    checkpoint = found[0].parent / "checkpoints" / name
    if not (checkpoint / "model.safetensors").is_file():
        raise FileNotFoundError(checkpoint / "model.safetensors")
    return checkpoint, progress


def predict(checkpoint, records, tokenizer):
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, local_files_only=True).to("cuda").eval()
    if model.config.label2id.get("ANSWERABLE") != 1:
        raise ValueError("wrong label mapping")
    probs = []
    for start in range(0, len(records), 8):
        batch = [r["text"] for r in records[start:start + 8]]
        inputs = tokenizer(batch, max_length=512, truncation=True, padding=True,
                           pad_to_multiple_of=8, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            probs.extend(torch.softmax(model(**inputs).logits.float(), dim=-1)[:, 1].cpu().tolist())
        if (start // 8 + 1) % 100 == 0 or start + len(batch) == len(records):
            print(json.dumps({"checkpoint": str(checkpoint), "done": start + len(batch)}), flush=True)
    del model
    torch.cuda.empty_cache()
    return probs


def score(y, p):
    tp = sum(a == b == 1 for a, b in zip(y, p))
    fp = sum(a == 0 and b == 1 for a, b in zip(y, p))
    fn = sum(a == 1 and b == 0 for a, b in zip(y, p))
    tn = len(y) - tp - fp - fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"rows": len(y), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "accuracy": (tp + tn) / len(y), "flag_rate_prediction_0": (tn + fn) / len(y)}


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required")
    raw_path, sro_path = BASE / "deberta_fg_dev.csv", SRO_DATA / "dev.csv"
    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:
        raise ValueError("dev CSV hash mismatch")
    raw, sro = csv_rows(raw_path), csv_rows(sro_path)
    for i, (a, b) in enumerate(zip(raw, sro)):
        if a["label"] != b["label"] or passage(a["text"]) != passage(b["text"]):
            raise ValueError(f"unpaired dev row {i}")
    checkpoints = {}
    progresses = {}
    for branch in ("raw", "sro"):
        checkpoints[branch], progresses[branch] = selected_checkpoint(branch)
    if OUT.exists():
        raise FileExistsError(OUT)
    OUT.mkdir(parents=True)
    tokenizer_root = checkpoints["sro"].parent / f"checkpoint-{progresses['sro']['global_step']}"
    for name in ("tokenizer.json", "tokenizer_config.json"):
        if not (tokenizer_root / name).is_file():
            raise FileNotFoundError(tokenizer_root / name)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_root, local_files_only=True)
    raw_probs = predict(checkpoints["raw"], raw, tokenizer)
    sro_probs = predict(checkpoints["sro"], sro, tokenizer)
    output = OUT / "paired_predictions.csv"
    with output.open("x", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(FIELDS)
        for i, (a, b) in enumerate(zip(raw, sro)):
            writer.writerow([i, a["label"], hashlib.sha256(a["text"].encode()).hexdigest().upper(),
                             hashlib.sha256(b["text"].encode()).hexdigest().upper(),
                             repr(raw_probs[i]), repr(sro_probs[i]),
                             int(raw_probs[i] >= 0.5), int(sro_probs[i] >= 0.5)])
    with output.open(encoding="utf-8", newline="") as file:
        paired = list(csv.DictReader(file))
    if len(paired) != 6404 or any(int(row["row_index"]) != i for i, row in enumerate(paired)):
        raise ValueError("paired readback failed")
    labels = [int(row["label"]) for row in paired]
    raw_preds = [int(row["raw_pred"]) for row in paired]
    sro_preds = [int(row["sro_pred"]) for row in paired]
    if AMBIGUOUS_INDICES is None:
        raise ValueError("frozen SRO status indices missing")
    ambiguous = set(AMBIGUOUS_INDICES)
    if len(ambiguous) != len(AMBIGUOUS_INDICES) or any(i < 0 or i >= 6404 for i in ambiguous):
        raise ValueError("invalid frozen SRO status indices")
    subsets = {}
    for status, indices in (("ok", [i for i in range(6404) if i not in ambiguous]),
                            ("ambiguous", sorted(ambiguous))):
        subsets[status] = {"rows": len(indices),
                           "raw": score([labels[i] for i in indices], [raw_preds[i] for i in indices]),
                           "sro": score([labels[i] for i in indices], [sro_preds[i] for i in indices])}
    disagreements = [i for i in range(6404) if raw_preds[i] != sro_preds[i]]
    error_slices = {}
    for name, preds in (("raw", raw_preds), ("sro", sro_preds)):
        error_slices[name] = {
            "false_sufficient_rows": [i for i in range(6404) if labels[i] == 0 and preds[i] == 1][:30],
            "false_insufficient_rows": [i for i in range(6404) if labels[i] == 1 and preds[i] == 0][:30]}
    report = {"status": "completed", "split_sha256": SPLIT_HASH,
              "raw_checkpoint": str(checkpoints["raw"]), "sro_checkpoint": str(checkpoints["sro"]),
              "raw_validation_f1": progresses["raw"]["best_validation_f1"],
              "sro_validation_f1": progresses["sro"]["best_validation_f1"],
              "raw": score(labels, raw_preds), "sro": score(labels, sro_preds),
              "paired_disagreement_rows": len(disagreements),
              "paired_disagreement_sample_indices": disagreements[:50],
              "coverage": {"ok": 6404 - len(ambiguous), "ambiguous_fallback": len(ambiguous)},
              "subsets": subsets, "error_slices": error_slices,
              "predictions_sha256": sha(output), "runtime": {"torch": torch.__version__,
              "transformers": transformers.__version__}}
    (OUT / "metrics.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 800}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 1600}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 2400}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 3200}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 4000}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_raw/checkpoints/checkpoint-2639", "done": 4800}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 800}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 1600}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 2400}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 3200}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 4000}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_sro/checkpoints/checkpoint-5278", "done": 4800}
{"checkpoint": "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/